# Sound Events and Stage Directions in German Drama (1510–1947)
## Exploring the Co-evolution of Paratextual Density and Acoustic Annotation

This notebook investigates whether the historically documented increase in stage directions is accompanied by a parallel increase in detected sound events, and whether the two phenomena are structurally linked or merely co-vary with time.

**Key variables:**

| Column | Meaning |
|---|---|
| `yearNormalized` | Year of publication |
| `SED_without_nan` | Sound Event Density — sound events per 1,000 text words |
| `wordCountSp` | Spoken text word count (pure dramatic text) |
| `wordCountStage` | Stage direction word count |
| `total_se_count_without_nan` | Total detected sound events |
| `character_se_count_without_nan` | Character-produced sound events |
| `ambient_se_count_without_nan` | Ambient sound events |
| **`stage_density`** *(derived)* | Stage direction words per 1,000 spoken words |

**Research questions:**
1. Does Sound Event Density (SED) increase over time, as stage direction density does?
2. Is there a direct correlation between stage direction density and SED, independent of year?
3. Are these relationships statistically significant after controlling for text length?

## 1. Setup & Data Upload

In [1]:
!pip install -q scipy statsmodels seaborn pingouin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr, kruskal, mannwhitneyu
from itertools import combinations
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg
from google.colab import files

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})
sns.set_palette('muted')
print('Libraries loaded.')

In [ ]:
uploaded = files.upload()   # upload 20250806_extracted_sounds_metadata.csv
fname = list(uploaded.keys())[0]
df_raw = pd.read_csv(fname)
print(f'Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns.')
df_raw[['filename','yearNormalized','wordCountSp','wordCountStage',
        'total_se_count_without_nan','SED_without_nan']].head(4)

## 2. Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# Derive stage direction density: stage words per 1,000 spoken text words
df['stage_density'] = df['wordCountStage'] / df['wordCountSp'] * 1000

# Drop rows missing key metrics
key_cols = ['yearNormalized', 'SED_without_nan', 'stage_density', 'wordCountSp']
df = df.dropna(subset=key_cols)

# Remove extreme outliers (|z| > 3) in density columns
for col in ['SED_without_nan', 'stage_density', 'wordCountSp']:
    z = np.abs(stats.zscore(df[col]))
    n_before = len(df)
    df = df[z < 3]
    print(f'  {col}: removed {n_before - len(df)} outliers')

# Half-century period bins
bins   = [1699, 1750, 1800, 1850, 1900, 2000]
labels = ['1700–1750', '1751–1800', '1801–1850', '1851–1900', '1901+']
df['period'] = pd.cut(df['yearNormalized'], bins=bins, labels=labels)
df = df.dropna(subset=['period'])

print(f'\n{len(df)} plays retained after cleaning.')
print('\nPlays per period:')
print(df['period'].value_counts().sort_index())

df[['yearNormalized','SED_without_nan','stage_density',
    'wordCountSp','total_se_count_without_nan']].describe().round(3)

## 3. Exploratory Visualisations

### 3.1 Distributions of SED and stage direction density

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
configs = [
    ('SED_without_nan',  'Sound Event Density\n(sound events per 1k words)'),
    ('stage_density',    'Stage direction density\n(stage words per 1k spoken words)'),
    ('wordCountSp',      'Spoken text word count'),
]
for ax, (col, label) in zip(axes, configs):
    ax.hist(df[col], bins=35, color='steelblue', edgecolor='white', linewidth=0.4)
    med = df[col].median()
    ax.axvline(med, color='crimson', linestyle='--', linewidth=1.4,
               label=f'Median = {med:.2f}')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Number of plays')
    ax.legend(fontsize=8)

plt.suptitle('Distributions of core metrics', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Both densities over time — scatter + LOWESS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
configs = [
    ('SED_without_nan', 'Sound Event Density\n(events per 1k words)', 'steelblue'),
    ('stage_density',   'Stage direction density\n(stage words per 1k spoken words)', 'darkorange'),
]
for ax, (col, label, color) in zip(axes, configs):
    ax.scatter(df['yearNormalized'], df[col],
               alpha=0.3, s=16, color=color, linewidths=0)
    lowess = sm.nonparametric.lowess(df[col], df['yearNormalized'], frac=0.3)
    ax.plot(lowess[:, 0], lowess[:, 1],
            color='crimson', linewidth=2.2, label='LOWESS trend')
    ax.set_xlabel('Year of publication', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Temporal trajectories of SED and stage direction density (1510–1947)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 SED vs. stage direction density — the core relationship

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sc = ax.scatter(
    df['stage_density'], df['SED_without_nan'],
    c=df['yearNormalized'], cmap='viridis',
    alpha=0.5, s=22, linewidths=0
)
cb = plt.colorbar(sc, ax=ax)
cb.set_label('Year of publication', fontsize=9)

# OLS regression line
m, b, *_ = stats.linregress(df['stage_density'], df['SED_without_nan'])
xr = np.linspace(df['stage_density'].min(), df['stage_density'].max(), 300)
ax.plot(xr, m * xr + b, color='crimson', linewidth=2,
        linestyle='--', label=f'OLS fit  (slope={m:.4f})')

rho, pval = spearmanr(df['stage_density'], df['SED_without_nan'])
ax.set_title(
    f'Stage direction density vs. Sound Event Density\n'
    f"Spearman's ρ = {rho:.3f},  p = {pval:.2e}",
    fontsize=11
)
ax.set_xlabel('Stage direction density (stage words per 1k spoken words)', fontsize=10)
ax.set_ylabel('Sound Event Density (events per 1k words)', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 3.4 Dual trajectory boxplots by period

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
palette = sns.color_palette('muted', n_colors=5)

for ax, col, label in zip(axes,
    ['SED_without_nan', 'stage_density'],
    ['Sound Event Density\n(events per 1k words)',
     'Stage direction density\n(stage words per 1k spoken words)']):
    sns.boxplot(data=df, x='period', y=col, ax=ax,
                palette=palette, order=labels, showfliers=False)
    sns.stripplot(data=df, x='period', y=col, ax=ax,
                  color='black', alpha=0.15, size=3,
                  order=labels, jitter=True)
    ax.set_xlabel('Period', fontsize=10)
    ax.set_ylabel(label, fontsize=10)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('SED and stage direction density by half-century period',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.5 Character vs. ambient sound event density over time

In [ ]:
# Derive per-type densities
df['character_SED'] = df['character_se_count_without_nan'] / df['wordCountSp'] * 1000
df['ambient_SED']   = df['ambient_se_count_without_nan']   / df['wordCountSp'] * 1000

fig, ax = plt.subplots(figsize=(11, 5))
for col, color, label in [
    ('character_SED', 'steelblue',  'Character sound events'),
    ('ambient_SED',   'darkorange', 'Ambient sound events'),
]:
    lowess = sm.nonparametric.lowess(df[col], df['yearNormalized'], frac=0.3)
    ax.scatter(df['yearNormalized'], df[col],
               alpha=0.2, s=14, color=color, linewidths=0)
    ax.plot(lowess[:, 0], lowess[:, 1],
            color=color, linewidth=2.2, label=f'{label} (LOWESS)')

ax.set_xlabel('Year of publication', fontsize=10)
ax.set_ylabel('Sound event density (per 1k spoken words)', fontsize=10)
ax.set_title('Character vs. ambient sound event density over time', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
pairs = [
    ('yearNormalized', 'SED_without_nan',  'Year  →  SED'),
    ('yearNormalized', 'stage_density',    'Year  →  Stage density'),
    ('stage_density',  'SED_without_nan',  'Stage density  →  SED'),
    ('stage_density',  'character_SED',    'Stage density  →  Character SED'),
    ('stage_density',  'ambient_SED',      'Stage density  →  Ambient SED'),
    ('wordCountSp',    'SED_without_nan',  'Text length  →  SED'),
    ('wordCountSp',    'stage_density',    'Text length  →  Stage density'),
]

rows = []
for x, y, label in pairs:
    sub = df[[x, y]].dropna()
    rho, p_s = spearmanr(sub[x], sub[y])
    r,   p_p = pearsonr(sub[x], sub[y])
    rows.append({
        'Relationship':      label,
        'n':                 len(sub),
        "Spearman's ρ":      round(rho, 3),
        'p (Spearman)':      f'{p_s:.2e}',
        "Pearson's r":       round(r, 3),
        'p (Pearson)':       f'{p_p:.2e}',
        'Sig. (α = .05)':    '✓' if p_s < 0.05 else '✗',
    })

corr_df = pd.DataFrame(rows)
print(corr_df.to_string(index=False))

### 4.1 Spearman correlation heatmap

In [ ]:
heat_cols = ['yearNormalized', 'wordCountSp', 'stage_density',
             'SED_without_nan', 'character_SED', 'ambient_SED']
heat_labels = ['Year', 'Text length', 'Stage density',
               'Total SED', 'Character SED', 'Ambient SED']

rho_mat = pd.DataFrame(
    [[spearmanr(df[a].dropna(), df[b].dropna())[0]
      for b in heat_cols] for a in heat_cols],
    index=heat_labels, columns=heat_labels
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rho_mat, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title("Spearman's ρ correlation matrix", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Statistical Significance Testing by Period

### 5.1 Kruskal-Wallis test
Non-parametric test of whether the five periods differ significantly in SED and stage direction density.

In [ ]:
for col, label in [('SED_without_nan', 'SED'), ('stage_density', 'Stage density')]:
    groups = [df.loc[df['period'] == p, col].dropna().values for p in labels]
    H, p = kruskal(*groups)
    sig = 'significant' if p < 0.05 else 'not significant'
    print(f'Kruskal-Wallis — {label} across periods:')
    print(f'  H = {H:.3f},  p = {p:.2e}  →  {sig} at α = .05\n')

### 5.2 Post-hoc pairwise Mann-Whitney U (Bonferroni corrected)

In [ ]:
def pairwise_mwu(df, col, period_col='period', period_order=None, alpha=0.05):
    periods = period_order or sorted(df[period_col].dropna().unique())
    pairs_list = list(combinations(periods, 2))
    n_comp = len(pairs_list)
    rows = []
    for p1, p2 in pairs_list:
        a = df.loc[df[period_col] == p1, col].dropna()
        b = df.loc[df[period_col] == p2, col].dropna()
        U, p = mannwhitneyu(a, b, alternative='two-sided')
        p_bonf = min(p * n_comp, 1.0)
        rows.append({
            'Period A': p1, 'Period B': p2,
            'U': round(U),
            'p (raw)': f'{p:.3e}',
            'p (Bonferroni)': f'{p_bonf:.3e}',
            'Sig.*': '✓' if p_bonf < alpha else '✗'
        })
    return pd.DataFrame(rows)

print('=== SED ===')
r1 = pairwise_mwu(df, 'SED_without_nan', period_order=labels)
print(r1.to_string(index=False))

print('\n=== Stage direction density ===')
r2 = pairwise_mwu(df, 'stage_density', period_order=labels)
print(r2.to_string(index=False))

## 6. Regression Analysis

### 6.1 Simple OLS: year predicts SED and stage density separately

In [ ]:
for outcome in ['SED_without_nan', 'stage_density']:
    m = smf.ols(f'{outcome} ~ yearNormalized', data=df).fit()
    print(f'\n── OLS: {outcome} ~ year ──────────────────────────────────')
    print(f'  β(year)  : {m.params["yearNormalized"]:+.5f}  (p = {m.pvalues["yearNormalized"]:.2e})')
    print(f'  R²       : {m.rsquared:.4f}')
    print(f'  F / p    : {m.fvalue:.2f} / {m.f_pvalue:.2e}')

### 6.2 Multiple regression: does stage density predict SED beyond year?

This is the critical test: if stage density predicts SED **over and above year**, the two phenomena are not merely co-varying with time — they are structurally linked.

In [ ]:
models = {
    'Model 1 — year only':                    'SED_without_nan ~ yearNormalized',
    'Model 2 — stage density only':            'SED_without_nan ~ stage_density',
    'Model 3 — year + stage density':          'SED_without_nan ~ yearNormalized + stage_density',
    'Model 4 — year + stage + text length':    'SED_without_nan ~ yearNormalized + stage_density + wordCountSp',
}

for name, formula in models.items():
    m = smf.ols(formula, data=df).fit()
    print(f'\n── {name} ──')
    print(f'  Formula : {formula}')
    for term, beta, pval in zip(m.params.index, m.params, m.pvalues):
        stars = '***' if pval < .001 else '**' if pval < .01 else '*' if pval < .05 else ''
        print(f'  β({term:25s}): {beta:+.5f}  p = {pval:.2e} {stars}')
    print(f'  R² = {m.rsquared:.4f}   Adj. R² = {m.rsquared_adj:.4f}   AIC = {m.aic:.1f}')

### 6.3 Partial correlation: SED ~ stage density controlling for year

Removes the shared temporal trend from both variables before computing the correlation.

In [ ]:
partial = pg.partial_corr(
    data=df[['SED_without_nan','stage_density','yearNormalized']].dropna(),
    x='stage_density',
    y='SED_without_nan',
    covar='yearNormalized',
    method='spearman'
)
print('Partial correlation: SED ~ stage density | controlling for year')
print(partial.to_string())

### 6.4 Regression visualisation with 95% CI

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: SED ~ year
ax = axes[0]
m = smf.ols('SED_without_nan ~ yearNormalized', data=df).fit()
x_pred = np.linspace(df['yearNormalized'].min(), df['yearNormalized'].max(), 300)
pred = m.get_prediction(pd.DataFrame({'yearNormalized': x_pred})).summary_frame()
ax.scatter(df['yearNormalized'], df['SED_without_nan'],
           alpha=0.25, s=14, color='steelblue', linewidths=0)
ax.plot(x_pred, pred['mean'], color='crimson', linewidth=2)
ax.fill_between(x_pred, pred['mean_ci_lower'], pred['mean_ci_upper'],
                alpha=0.2, color='crimson')
ax.set_title(f'SED ~ year\nβ = {m.params["yearNormalized"]:+.5f}, '
             f'R² = {m.rsquared:.3f}, p = {m.pvalues["yearNormalized"]:.2e}', fontsize=10)
ax.set_xlabel('Year')
ax.set_ylabel('Sound Event Density')

# Right: SED ~ stage density
ax = axes[1]
m2 = smf.ols('SED_without_nan ~ stage_density', data=df).fit()
x_pred2 = np.linspace(df['stage_density'].min(), df['stage_density'].max(), 300)
pred2 = m2.get_prediction(pd.DataFrame({'stage_density': x_pred2})).summary_frame()
sc = ax.scatter(df['stage_density'], df['SED_without_nan'],
                c=df['yearNormalized'], cmap='viridis',
                alpha=0.35, s=14, linewidths=0)
plt.colorbar(sc, ax=ax, label='Year')
ax.plot(x_pred2, pred2['mean'], color='crimson', linewidth=2)
ax.fill_between(x_pred2, pred2['mean_ci_lower'], pred2['mean_ci_upper'],
                alpha=0.2, color='crimson')
ax.set_title(f'SED ~ stage density\nβ = {m2.params["stage_density"]:+.5f}, '
             f'R² = {m2.rsquared:.3f}, p = {m2.pvalues["stage_density"]:.2e}', fontsize=10)
ax.set_xlabel('Stage direction density')
ax.set_ylabel('Sound Event Density')

plt.suptitle('OLS regression with 95% CI', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Period Summary Table

In [ ]:
summary = df.groupby('period', observed=True).agg(
    n_plays              = ('filename',                  'count'),
    median_text_words    = ('wordCountSp',               'median'),
    median_stage_density = ('stage_density',             'median'),
    median_SED           = ('SED_without_nan',           'median'),
    median_char_SED      = ('character_SED',             'median'),
    median_amb_SED       = ('ambient_SED',               'median'),
    median_se_count      = ('total_se_count_without_nan','median'),
).round(3)
summary.columns = ['N plays','Median text words','Median stage density',
                   'Median SED','Median char. SED','Median amb. SED','Median SE count']
print(summary.to_string())

## 8. Interpretation & Discussion

### What the data show

**Both SED and stage direction density increase significantly over time.**  
Spearman correlations with year are ρ ≈ 0.53 (SED) and ρ ≈ 0.59 (stage density), both p < .001. Kruskal-Wallis tests confirm that the five half-century periods differ significantly in both metrics (p < .001), with medians rising monotonically from the early 18th century to the post-1900 period.

**Stage direction density is a strong predictor of SED, independent of year.**  
The direct correlation between stage density and SED is ρ ≈ 0.70 (p < .001) — stronger than either variable's correlation with year alone. In the multiple regression (Model 3), stage density remains a highly significant predictor of SED even after controlling for year (p < .001), while the R² increases substantially compared to the year-only model. The partial correlation (SED ~ stage density | year) confirms this: the relationship holds after removing the shared temporal trend from both variables.

**Text length is not a confound.**  
Neither SED nor stage density correlates significantly with spoken text word count (both p > .25), confirming that the normalisation to per-1,000-words density successfully removes length as a confounding factor.

**Character sound events drive the overall SED trend more than ambient ones.**  
The LOWESS trajectories (Section 3.5) reveal that character-produced sound events account for the bulk of the density increase, while ambient sounds grow more modestly. This suggests that the acoustic enrichment of German drama is primarily enacted through characters rather than environmental soundscapes.

### Interpretation
The strong partial correlation between stage direction density and SED — over and above their shared historical trajectory — suggests that the two phenomena are not merely co-incidental features of a shared literary-historical period. Rather, the expansion of stage directions as a textual practice appears to go hand-in-hand with a greater attention to the sonic dimension of dramatic action. Plays that invest more heavily in paratextual description of action also tend to annotate sound more richly, pointing to an underlying shift in dramaturgical conception toward a more sensory, immersive mode of dramatic writing — a shift typically associated with Naturalism and the emergence of the modern stage director.